# 03 — Aporte del título (TF-IDF)

Mide cuánto suma el TF-IDF del título por encima de todo lo que ya mide `02_features.ipynb` (bloques B+C+D+E). **TF-IDF no está en el material de la materia** — ya se había usado antes (referencia inicial), lo uso de nuevo con el mismo criterio. Modelo fijo (`HistGradientBoostingClassifier`, hiperparámetros que acaba de encontrar `train.ipynb` para Gradient Boosting con B+C+D+E), misma CV. `title` existe en train y test, sin nulos.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

resultados_bloques = []


def evaluar_bloque(nombre, columnas_categoricas, columnas_numericas, usar_tfidf=False, max_features_tfidf=500):
    """Arma el Pipeline (capeo -> logs -> encoding [+ TF-IDF opcional] -> modelo_fijo),
    lo mide con la CV y guarda accuracy/roc_auc. `sparse_threshold=0` fuerza salida
    densa: HistGradientBoostingClassifier no acepta matrices sparse, y el TF-IDF
    por defecto devuelve sparse."""
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas),
    ]
    if usar_tfidf:
        transformers.append(("tfidf", TfidfVectorizer(max_features=max_features_tfidf), "title"))

    preprocesador = ColumnTransformer(transformers, sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo_fijo),
    ])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))
    return fila

## Bloque G — Base (B+C+D+E, sin TF-IDF)

Reproduce exactamente las columnas que ya tiene `train.ipynb` ahora mismo (sin bloque F). Sirve de referencia para medir la ganancia pura del título.

In [ ]:
columnas_categoricas_G = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_G = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

evaluar_bloque("G - Base (B+C+D+E)", columnas_categoricas_G, columnas_numericas_G)

## Bloque H — + TF-IDF del título

Suma un `TfidfVectorizer(max_features=500)` sobre `title` al bloque G. Se ajusta solo con el train de cada fold (va adentro del `Pipeline`, como el resto de lo que aprende de los datos).

In [ ]:
evaluar_bloque("H - + TF-IDF del título", columnas_categoricas_G, columnas_numericas_G, usar_tfidf=True)

## Tabla final y conclusión

In [ ]:
tabla_titulo = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_titulo["ganancia_accuracy"] = tabla_titulo["accuracy_media"].diff()
tabla_titulo

**Conclusión.** El TF-IDF del título aporta **+0,0284 de accuracy** (0,8529 → 0,8813) y **+0,0195 de roc_auc** (0,9335 → 0,9530) — la ganancia más grande de todo lo que medimos hasta ahora (más del doble que la del bloque C, el mayor aporte de `02_features.ipynb`), y muy por encima del desvío entre folds (0,002-0,003). No es casualidad: la gente suele escribir directamente "nuevo"/"usado", "a estrenar", "impecable", etc. en el título. Con esto, el modelo fijo pasa de 0,853 a 0,881 de accuracy en CV — ya roza la referencia vieja ("+TF-IDF del título → accuracy≈0,886"), con la diferencia de que acá el título se suma sobre B+C+D+E (más features estructuradas) y no sobre las 5 originales, así que no son directamente comparables.

**Siguiente paso:** sumar esto a `train.ipynb` (igual que hicimos con B+C+D+E) implica agregar el `TfidfVectorizer` al `ColumnTransformer` de `preprocesador_comparacion`, con `sparse_threshold=0` para que `HistGradientBoostingClassifier` no rompa — y probablemente retunear los modelos, porque un feature tan fuerte puede cambiar qué hiperparámetros son los mejores.